In [ ]:
%matplotlib inline
import seaborn as sns
import pandas as pd
import numpy as np
import sklearn
import sklearn.datasets
import sklearn.linear_model
from sklearn import preprocessing
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
scaler = StandardScaler()
#scaler = MinMaxScaler()
label_e=preprocessing.LabelEncoder()
import matplotlib.pyplot as plt
import sklearn.metrics as metrics
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVR, LinearSVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import GradientBoostingRegressor
#sns.set()

In [ ]:
census_s = pd.read_csv('/kaggle/input/godaddy-microbusiness-density-forecasting/census_starter.csv')
sample_s = pd.read_csv('/kaggle/input/godaddy-microbusiness-density-forecasting/sample_submission.csv')
test_df = pd.read_csv('/kaggle/input/godaddy-microbusiness-density-forecasting/test.csv')
train_df = pd.read_csv('/kaggle/input/godaddy-microbusiness-density-forecasting/train.csv')

In [ ]:
## Function for Checking Empty Values
def empty(data):
    plt.figure(figsize=(10,10))
    sns.heatmap(data.isnull(),yticklabels=False,cbar=False,cmap='viridis')
    
## Function for Checking Correlation among features
def cor_map(data):
    corrmat = pd.DataFrame(data).corr(method = "pearson")
    plt.figure(figsize=(15,15))
    #plot heat map
    g=sns.heatmap(corrmat,annot=True)

In [ ]:
empty(train_df) ## No Null Values Found

In [ ]:
empty(census_s)

In [ ]:
census_s[census_s.isna().any(axis=1)]

In [ ]:
## Filling census null values
for col in census_s.columns:
    census_s[col].fillna(census_s[col].mean(),inplace=True)
census_s.head()

In [ ]:
cor_map(census_s)
## Colinearity among features is observed

In [ ]:
def calc_reg_return_vif(X, y):
    """
    Utility function to calculate the VIF. This section calculates the linear
    regression inverse R squared.

    Parameters
    ----------
    X : DataFrame
        Input data.
    y : Series
        Target.

    Returns
    -------
    vif : float
        Calculated VIF value.

    """
    X = X.values
    y = y.values

    if X.shape[1] == 1:
        print("Note, there is only one predictor here")
        X = X.reshape(-1, 1)
    reg = LinearRegression().fit(X, y)
    vif = 1 / (1 - reg.score(X, y))

    return vif


def calc_vif(df):
    """
    Calculating VIF using function from scratch

    Parameters
    ----------
    df : DataFrame
        without target variable.

    Returns
    -------
    vif : DataFrame
        giving the feature - VIF value pair.

    """

    vif = pd.DataFrame()

    vif_list = []
    for feature in list(df.columns):
        y = df[feature]
        X = df.drop(feature, axis="columns")
        vif_list.append(calc_reg_return_vif(X, y))
    vif["feature"] = df.columns
    vif["VIF"] = vif_list
    print(vif)
    return vif

In [ ]:
## Let's have a look at the VIF scores for the features
census_vif = calc_vif(census_s) 

In [ ]:
#PCA being done to counter colinearity and reduce data dimension
pca = PCA(n_components = 8)
pca.fit(census_s.drop(columns=['cfips'],axis=1))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance');

In [ ]:
new_census_s = pd.DataFrame(scaler.fit_transform(PCA(8).fit_transform(census_s.drop(columns=['cfips'],axis=1))))
#new_census_s = pd.DataFrame(PCA(8).fit_transform(census_s.drop(columns=['cfips'],axis=1)))
new_census_s['cfips'] = census_s['cfips']
census_vif = calc_vif(new_census_s)

In [ ]:
new_census_s.head()

In [ ]:
drop_cols = ['county','state','active','first_day_of_month','microbusiness_density','row_id']
y = train_df[['microbusiness_density']]

## Prep Function for Data
def prep(data):
    data1 = data.copy()
    data1.drop_duplicates(inplace=True)
    data1['date'] = pd.to_datetime(data1['first_day_of_month']).dt.day
    data1['month'] = pd.to_datetime(data1['first_day_of_month']).dt.month
    data1['year'] = pd.to_datetime(data1['first_day_of_month']).dt.year
    for col in drop_cols:
        try:
            data1.drop(columns = [col],axis=1,inplace=True)
        except:
            print(col," not found")
    data1 = data1.merge(new_census_s,how='left',on='cfips')
    data1.columns = data1.columns.astype(str)
    print('old data shape: ',data.shape)
    print('new data shape: ',data1.shape)
    return data1

In [ ]:
new_train_df = prep(train_df)

In [ ]:
new_train_df.head()

In [ ]:
train_df_vif = calc_vif(new_train_df)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(new_train_df,y,random_state=0)

In [ ]:
print(x_train.shape)
print(y_train.shape)

In [ ]:
# estimators = sklearn.utils.all_estimators(type_filter=None)
# from sklearn import base
# for name, class_ in estimators:
#     if issubclass(class_, base.RegressorMixin):
#         print(name)

In [ ]:
## Cost Function
def smape(A, F):
    #print(A.shape)
    #print(F.shape)
    tmp = 2 * np.abs(F - A) / (np.abs(A) + np.abs(F))
    len_ = np.count_nonzero(~np.isnan(tmp))
    if len_ == 0 and np.nansum(tmp) == 0: # Deals with a special case
        return 100
    return 100 / len_ * np.nansum(tmp)

In [ ]:
## Creating models list

models = []

models.append(("LinearRegression",LinearRegression()))
models.append(("KNeighbors",KNeighborsRegressor(n_neighbors=4)))
models.append(("DecisionTree",DecisionTreeRegressor()))
models.append(("XGBRegressor",XGBRegressor()))
models.append(("RandomForest",RandomForestRegressor()))
models.append(("AdaBoostRegressor",AdaBoostRegressor(n_estimators=100, random_state=0)))
#models.append(("MLPRegressor",MLPRegressor(solver='lbfgs', random_state=0,max_iter=10000)))
models.append(("BaggingRegressor",BaggingRegressor(n_estimators=10, random_state=0)))
models.append(("GradBoostingRegressor",GradientBoostingRegressor(n_estimators=120, learning_rate=0.05,max_depth=8, random_state=0)))
#print(models)

train_score = []
names = []
test_score = []

In [ ]:
%%time

np.seterr(divide='ignore', invalid='ignore')
for name,model in models:
    print("For model : ",name)
    m = model.fit(x_train,y_train.values.ravel())
    names.append(name)
    trs = smape(y_train.values.ravel(),m.predict(x_train))
    tes = smape(y_test.values.ravel(),m.predict(x_test))
    train_score.append(trs)
    test_score.append(tes)
    print("Training Score : ",trs)
    print("Testing Score : ",tes)


In [ ]:
%%time

## Example submission

m = RandomForestRegressor()
final_train_df = prep(train_df)
final_test_df = prep(test_df)
model=m.fit(final_train_df,y.values.ravel())
trs = smape(y.values.ravel(),m.predict(final_train_df))
print('training_score: ',trs)
pred = test_df[['row_id']].copy()
pred['microbusiness_density'] = model.predict(final_test_df)
print(pred.shape)
print(pred.head())
pred.to_csv('submission.csv',index=False)